In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:

IMG_SIZE = (224,224)
BATCH_SIZE = 32
DATASET_DIR = 'leaf_dataset_small'

In [4]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 1600 images belonging to 2 classes.
Found 400 images belonging to 2 classes.


In [5]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

model.save("model/leaf_detector.keras")

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 44s 832ms/step - accuracy: 0.9438 - loss: 0.1328 - val_accuracy: 0.9500 - val_loss: 0.2021
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 61s 414ms/step - accuracy: 0.9944 - loss: 0.0295 - val_accuracy: 0.9475 - val_loss: 0.1955
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 417ms/step - accuracy: 0.9931 - loss: 0.0188 - val_accuracy: 0.9575 - val_loss: 0.1837
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 415ms/step - accuracy: 0.9975 - loss: 0.0117 - val_accuracy: 0.9550 - val_loss: 0.1994
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 406ms/step - accuracy: 1.0000 - loss: 0.0045 - val_accuracy: 0.9575 - val_loss: 0.1933
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 415ms/step - accuracy: 0.9987 - loss: 0.0043 - val_accuracy: 0.9475 - val_loss: 0.2864
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 409ms/step - accuracy: 1.0000 - loss: 0.0028 - val_accuracy: 0.9500 - val_loss: 0.2731
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 405ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accu

In [8]:
import numpy as np
from PIL import Image
img = Image.open("test_image.jfif")
img = img.resize((224,224))

img_array = np.array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)[0][0]

label = "not_leaf" if pred > 0.5 else "leaf"

print("Prediction:", label)
print("Confidence:", float(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 617ms/step
Prediction: leaf
Confidence: 0.008131347596645355


In [9]:
print(train_generator.class_indices)

{'leaf': 0, 'not_leaf': 1}


In [10]:
model.export("model/leaf_model_serving")

INFO:tensorflow:Assets written to: model/leaf_model_serving\assets


INFO:tensorflow:Assets written to: model/leaf_model_serving\assets


Saved artifact at 'model/leaf_model_serving'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2635723089936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723089168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723091088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723091856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723092048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723088016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723091280: TensorS

In [11]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("cipherplant.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpsq9a6q11\assets


INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpsq9a6q11\assets


Saved artifact at 'C:\Users\admin\AppData\Local\Temp\tmpsq9a6q11'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2635723089936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723089168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723091088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723090128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723091856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723092048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635723088016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2